# Gold table


In [ ]:
from pathlib import Path

import pandas as pd

EXPERIMENT = "gulf_stream_pigment_influencers_20241001_20251231"
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
gold_table_path = repo_root / "data" / EXPERIMENT / "gold" / "eddy_pigment_table.parquet"

gold = pd.read_parquet(gold_table_path)
print(gold)

In [ ]:
gold.columns

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

target_cols = [x for x in list(gold.columns) if x.startswith("log_ratio")]
predictor_cols = ["polarity", "season", "movement", "age_frac"]
X = pd.get_dummies(gold[predictor_cols])
y = gold[target_cols]
# Remove rows w/ nans since that messes up Random Forest Regression
mask = y.notna().all(axis=1) # notna() returns same shape, .all(axis=1) collapses across columns returning a series of bools of length (rows)
X, y = X[mask], y[mask]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026)

# group one-hot columns back under their predictor (season_DJF -> season)
col_to_group = {c: next(p for p in predictor_cols if c == p or c.startswith(p + "_")) for c in X.columns}

models = {}
importances = {}
for col in target_cols:
    model = RandomForestRegressor(random_state=2026)
    model.fit(X_train, y_train[col])
    models[col] = model
    imp = pd.Series(model.feature_importances_, index=X.columns)
    importances[col] = imp.groupby(col_to_group).sum().sort_values(ascending=False)

for target, model in models.items():
    test_r2 = model.score(X_test, y_test[target])
    print(f"{target} R^2 value: {test_r2}")

print("---------------------------------------------------------------")

for col in target_cols:
    print(f"{col} predictor importances:")
    print(importances[col])

# A stronger forest

The forest above tops out around R^2 ~0.15, and goes negative for DV_chla.
Two things hold it back: the default `RandomForestRegressor` grows unbounded trees that memorize individual eddy-days, and the four predictors leave out each eddy's strength, size, and position.

Below is the same one-forest-per-pigment setup with the trees regularized and those features added.
Scoring also switches to GroupKFold by `track_id`: only 92 eddies make up the ~1290 eddy-days, so a random split would put the same eddy in train and test and inflate the result.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import r2_score

target_cols = [c for c in gold.columns if c.startswith("log_ratio")]
predictor_cols = [
    # class and lifecycle
    "polarity", "season", "movement", "age_frac",
    # strength, size, rotation
    "amplitude_cm", "radius_km", "age_days",
    "rossby_center", "rossby_abs_mean", "rossby_min", "rossby_max",
    # where it sits
    "gs_dist_km", "center_lat", "center_lon",
]
X = pd.get_dummies(gold[predictor_cols])
y = gold[target_cols].replace([np.inf, -np.inf], np.nan)
mask = y.notna().all(axis=1)  # drop the two eddy-days whose eddy mean is 0
X, y, groups = X[mask], y[mask], gold.loc[mask, "track_id"]

# 92 eddies make up these eddy-days, so a random split would leak the same eddy
# into train and test. GroupKFold keeps every eddy on one side of each split.
rf = RandomForestRegressor(random_state=2026, n_estimators=400,
                           max_depth=8, min_samples_leaf=3, max_features=0.5)
gkf = GroupKFold(n_splits=5)
r2 = pd.Series({
    col: r2_score(y[col], cross_val_predict(rf, X, y[col], groups=groups, cv=gkf, n_jobs=-1))
    for col in target_cols
}).sort_values(ascending=False)
print(r2.round(3).to_string())
print(f"mean R^2: {r2.mean():.3f}")

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit

# Impurity importance is free from the fit but inflates predictors with many split
# points (movement's four classes, the season one-hots). Permutation importance
# shuffles each predictor on held-out eddies and measures the real drop in R^2, so
# reading the two side by side is more honest than trusting either alone.
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=2026).split(X, y, groups))
col_to_group = {c: next(p for p in predictor_cols if c == p or c.startswith(p + "_")) for c in X.columns}

impurity_scores, perm_scores = [], []
for col in target_cols:
    fit = rf.fit(X.iloc[train_idx], y[col].iloc[train_idx])
    impurity_scores.append(pd.Series(fit.feature_importances_, index=X.columns).groupby(col_to_group).sum())
    pi = permutation_importance(fit, X.iloc[test_idx], y[col].iloc[test_idx],
                                n_repeats=10, random_state=2026, n_jobs=-1)
    perm_scores.append(pd.Series(pi.importances_mean, index=X.columns).groupby(col_to_group).sum())

importances = pd.DataFrame({
    "impurity": pd.concat(impurity_scores, axis=1).mean(axis=1),
    "permutation": pd.concat(perm_scores, axis=1).mean(axis=1),
}).sort_values("permutation", ascending=False)
print(importances.round(3).to_string())

Mean out-of-fold R^2 is now about 0.48, even under the stricter grouped scoring.
Most of the jump is the regularization (shallower trees, `max_features=0.5`) reining in the overfitting; the added features, mainly the position ones, supply the rest.

The two importance columns are meant to be read against each other.
Impurity importance is what the trees split on, and it flatters predictors with many split points, so `movement` (four classes) looks like a top-three driver.
Permutation importance instead shuffles each predictor on held-out eddies and measures the actual drop in R^2; by that measure distance to the Gulf Stream and season lead, latitude follows, and `movement`, `polarity`, eddy strength, and size all collapse toward zero.

So the eddy's own class barely helps: dropping both `polarity` and `movement` leaves held-out R^2 unchanged (~0.45), so the earlier "polarity matters" read was mostly impurity bias.
One nuance: permutation importance is marginal given the rest, so strength and size reading near zero partly means position already carries their information, not that they are meaningless on their own.

The standing caveat: position encodes the regional pigment gradient across the Gulf Stream front, not necessarily the eddy's own effect, and the log-ratio already divides by the local background.
A thread to pull on, not a verdict.